# TP 7-8 : jointures réparties

date de modification: 02/04/2026

# Introduction

On considère un SGBD réparti composé d'un SGBD local S0 et de
2 sources distantes : S1 et S2.

L'objectif de ce TP est d'étudier l'évaluation des requêtes accédant à des données situées dans plusieurs sources. Il s'agit de comprendre l'évaluation d'une requête de jointure entre 2 relations qui sont situées sur deux sources  distinctes.

*   Définir un schéma global décrivant l'ensemble des données et sur lequel un utlisateur peut formuler une requête globale.
*   Implanter l'évaluation répartie d'une requête globale
*   Comprendre l'ordre et l'emplacement des opérations permettant d'évaluer une requête répartie : savoir expliquer quel site traite quelles opérations.


# Préparation

installer (ou mettre à jour) DuckDB


In [ ]:
# duckdb est déjà installé sur les machines de colab
# !pip install duckdb --pre
#!pip install -U duckdb

installer un parser SQL

In [ ]:
pip install sqlglot

Affichage des résultats

In [2]:
pip install -q itables

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 13.2 MB/s eta 0:00:00


In [3]:
# pour améliorer l'affichage des tableaux contenant le resultat d'une requete
from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)

Les lib nécessaires pour ce TP

In [4]:
import duckdb
# import sqlite3
import time
import os

import pandas as pd
import os
import shutil


Les dossiers servant de zone de **transfert de données** entre les sites

In [5]:
local_dir = "tmp"
os.makedirs(os.path.join(local_dir, "S0"), exist_ok=True)
os.makedirs(os.path.join(local_dir, "S1"), exist_ok=True)
os.makedirs(os.path.join(local_dir, "S2"), exist_ok=True)
os.listdir(local_dir)

['S0', 'S1', 'S2']

Connections aux sources. Chaque source est un SGBD indépendant

In [6]:
S0 = duckdb.connect('base0.db')
S1 = duckdb.connect('base1.db')
S2 = duckdb.connect('base2.db')

# Créer les données


## Site S1 : Garage, Personne

On crée les tables du site S1

In [7]:
create_tables_1 = """
    drop table if exists Garage cascade;
    drop table if exists Personne cascade;

    drop sequence if exists id_garage;
    drop sequence if exists id_pers;

    CREATE SEQUENCE if not exists id_garage START 1;
    CREATE SEQUENCE if not exists id_pers START 1;

    CREATE TABLE if not exists Garage (
    idgarage INT DEFAULT nextval('id_garage'),
    nom VARCHAR(255),
    ville VARCHAR(255),
    jourdefermeture VARCHAR(255),
    PRIMARY KEY (idgarage)
    );

    CREATE TABLE if not exists Personne (
    idpers INT DEFAULT nextval('id_pers'),
    nom VARCHAR(255) NOT NULL,
    prenom VARCHAR(255),
    age INT,
    PRIMARY KEY (idpers)
    );
"""
S1.execute(create_tables_1)

In [8]:
insertion_1 = """
INSERT INTO Garage (nom, ville, jourdefermeture) VALUES ('Speedy', 'Paris', 'Dimanche'),
('AutoService', 'Tours', 'Lundi'), ('Feu Vert', 'Paris', 'Dimanche'),
('Garage lyonnais', 'Lyon', 'Samedi'), ('Modern Garage', 'Bordeaux', 'Lundi'),
('Garage Auto', 'Paris', 'Dimanche'), ('La Centrale', 'Marseille', 'Lundi'),
('AD Garage', 'Paris', 'Dimanche'), ('Euromaster', 'Lyon', 'Samedi'),
('Nauroto', 'La Rochelle', 'Dimanche');

INSERT INTO Personne (nom, prenom, age) VALUES ('Armand', 'Rébecca', 24),
('Hebert', 'Aimée', 35), ('Ribeiro', 'Marielle', 18), ('Savary', 'Hilaire', 27),
('Dupont', 'Jean', 64), ('Dubois', 'Léo', 23), ('Mbappé', 'Kylian', 21),
('Georges', 'Antoine', 25), ('Cavani', 'Edinson', 32),
('Messi', 'Lionel', 33), ('Precieuse', 'Pierre', 45),
('Durand','Jeanne', 18), ('Zoulou','Zoé', 18),
('Pitaut','Enzo', 18), ('Hunter','Paulo', 18);
"""

S1.execute(insertion_1)

Tables de S1

In [9]:
S1.execute("select table_name from duckdb_tables()").fetchall()

[('Garage',), ('Personne',)]

Shéma de S1

In [ ]:
# S1.execute("describe Garage").df()

In [10]:
shema_base = """
  SELECT t.table_name, c.column_name
  FROM duckdb_columns() c
  JOIN duckdb_tables() t ON (c.table_name = t.table_name)
  ORDER BY t.table_name
"""

S1.execute(shema_base).df()

Loading ITables v2.7.3 from the internet... (need help?)


In [ ]:
S1.execute("select * from Garage").df()

Loading ITables v2.7.3 from the internet... (need help?)


In [ ]:
S1.execute("select * from Personne").df()

Loading ITables v2.7.3 from the internet... (need help?)


## Site 2 : Mecanicien

In [11]:
create_tables_2 = """
drop table if exists Mecanicien cascade;

CREATE TABLE if not exists Mecanicien (
    idpers INT,
    idgarage INT NOT NULL,
    niveau INT,
    PRIMARY KEY (idpers),
    );
"""
S2.execute(create_tables_2)

In [12]:
insertion_2 = """
INSERT INTO Mecanicien (idpers, idgarage, niveau) VALUES
(1, 1, 2), (2, 1, 3), (4, 10, 2), (5, 10, 1), (6, 3, 2), (8, 6, 2),
(10, 3, 1), (11, 4, 2), (12, 4, 2), (13, 1, 2);
 """

S2.execute(insertion_2)

In [13]:
S2.execute(shema_base).df()

Loading ITables v2.7.3 from the internet... (need help?)


In [14]:
S2.execute("select * from Mecanicien").df()

Loading ITables v2.7.3 from the internet... (need help?)


## SQL Parser

Le parseur SQL vous permet d'analyser l'expression SQL d'une requête pour obtenir facilement le nom des tables accédées par la requête, la liste des prédicats de sélection et de jointure ainsi que les attributs projetés.

Pour faciliter le TP, veillez à écrire les requêtes SQL en respectant les règles suivantes:

*    Toujour utiliser la notation pointée: alias.attribut pour mentionner un attribut, ceci dans le SELECT, le WHERE, et le ON.
*    Dans le FROM : définir un alias pour chaque table
*    Définir les jointures avec les mots JOIN ON


In [15]:
from sqlglot import parse_one, exp
from sqlglot.expressions import Column, Select, Join, Where, Table, EQ, LT, LTE, GT, GTE, NEQ, Between, Literal, And, Or, Not
from sqlglot.optimizer.normalize import normalize

class Query:
  def __init__(self, query):
    self.table_names = list()
    self.table_aliases = {}
    self.projections = list()
    self.joins = list()
    self.selections = list()
    # self.query_ast = parse_one(query)
    self.query_ast = normalize(parse_one(query))

    # tables dans la clause FROM
    for table in self.query_ast.find_all(Table):
      self.table_names.append(table.name)
      if table.alias is not None:
        self.table_aliases[table.alias] = table.name

    # projections dans la clause SELECT
    for attr in self.query_ast.expressions:
      table = self.table_aliases.get(attr.table, attr.table)
      self.projections.append( (table, attr.name))

    # jointures dans la clause JOIN

    # # ancienne version ?
    # for j in self.query_ast.args['joins']:
    #   pred = j.args['on']
    #   left_table = self.table_aliases.get(pred.left.table,pred.left.table)
    #   right_table = self.table_aliases.get(pred.right.table,pred.right.table)
    #   self.joins.append( ( pred.sql(), left_table, pred.left.name, right_table, pred.right.name))

    for j in self.query_ast.find_all(Join):
      join_conds = []
      join_tables = set()
      if j.on(): # jointure avec le mot clé ON (donc ce n'est pas un natural join)
        condition = j.args.get("on")
        if isinstance(condition, And):  # Si conjonction (AND) de plusieurs predicats
            conditions = condition.flatten()
        else:
            conditions = [condition]

        for expr in conditions:
            join_conds.append(expr.sql())
            # Extraire les tables des colonnes impliquées
            for col in expr.find_all(Column):
                join_tables.add(col.table)
        self.joins.append( ( join_conds, join_tables))

      # pred = j.args['on']
      # left_table = self.table_aliases.get(pred.left.table,pred.left.table)
      # right_table = self.table_aliases.get(pred.right.table,pred.right.table)
      # self.joins.append( ( pred.sql(), left_table, pred.left.name, right_table, pred.right.name))
      # # if isinstance(pred, EQ):
      # #   print("equi jointure")
      # # else:
      # #   print("theta jointure")


    # sélections dans la clause WHERE
    # Liste des types autorisés
    COMPARISON_OPS = (EQ, LT, LTE, GT, GTE, NEQ)

    for pred in self.query_ast.find(exp.Where).find_all(exp.Predicate):
      if isinstance(pred, COMPARISON_OPS) :
        left_table = self.table_aliases.get(pred.left.table, pred.left.table)
        if isinstance(pred.right, Literal):
          self.selections.append( (pred.sql(), left_table) )
        elif isinstance(pred.right, Column):
          right_table = self.table_aliases.get(pred.right.table,pred.right.table)
          self.joins.append( ( pred.sql(), left_table, pred.left.name, right_table, pred.right.name))
          # if isinstance(pred, EQ):
          #   print("équi-jointure")
          # else:
          #   print("théta-jointure")
        else:
          print("predicat non pris en compte dans le TP !!! ", pred )
      elif isinstance(pred, exp.Between) or isinstance(pred, exp.In):
        table = self.table_aliases.get(pred.this.table, pred.this.table)
        self.selections.append( (pred.sql(), table))
      # elif isinstance(pred, exp.In):
      #   table = self.table_aliases.get(pred.this.table, pred.this.table)
      #   pred = pred.args
      else:
        print("predicat non pris en compte dans le TP !!! ", pred )

  def get_tables(self):
      return self.table_names

  def get_aliases(self):
      return self.table_aliases

  def get_projections(self):
      return self.projections

  def get_joins(self):
      return self.joins

  def get_selections(self):
        return self.selections

  def get_ast(self):
        return self.query_ast

  def get_ast_string(self):
        return repr(self.query_ast)


print("class Query defined")


R1 = """
SELECT g.nom, g.ville
FROM Garage g
JOIN Mecanicien m ON g.idgarage = m.idgarage
WHERE m.niveau = 2
"""

# print(repr(parse_one(R1)))

query = Query(R1)
print("table names:", query.get_tables())
print("table aliases:", query.get_aliases())
print("projections:", query.get_projections())
print("joins:", query.get_joins())

class Query defined
table names: ['Garage', 'Mecanicien']
table aliases: {'g': 'Garage', 'm': 'Mecanicien'}
projections: [('Garage', 'nom'), ('Garage', 'ville')]
joins: [(['g.idgarage = m.idgarage'], {'m', 'g'})]


Exemple d'utilisation du parseur SQL pour R1

In [16]:
R1 = """
SELECT g.nom, g.ville
FROM Garage g
JOIN Mecanicien m ON g.idgarage = m.idgarage
WHERE m.niveau = 2
"""

# R2 = """
# SELECT g.nom, g.ville
# FROM Garage g
# JOIN Mecanicien m ON g.idgarage = m.idgarage
# WHERE (m.niveau between 1 and 2) AND m.idpers <> 10
# """

# R3 = """
# SELECT g.nom, g.ville
# FROM Garage g
# JOIN Mecanicien m ON g.idgarage = m.idgarage
# WHERE m.niveau in (1,2)
# """

query = Query(R1)

# print("arbre syntaxique de la requete", query.get_ast_string())

print("table names:", query.get_tables())
print("table aliases:", query.get_aliases())
print("projections:", query.get_projections())
print("joins:", query.get_joins())
print("selections:", query.get_selections())

table names: ['Garage', 'Mecanicien']
table aliases: {'g': 'Garage', 'm': 'Mecanicien'}
projections: [('Garage', 'nom'), ('Garage', 'ville')]
joins: [(['g.idgarage = m.idgarage'], {'m', 'g'})]
selections: [('m.niveau = 2', 'Mecanicien')]


# Exercice 1

Proposer une solution pour traiter les requêtes suivantes, telle que :

*  les sous-requêtes sont traitées sur S1 ou S2 autant que possible,

*  les résultats intermédiaires peuvent être transférés dans S0 pour obtenir le résultat final.


## Question 1

La requete globale R1 est

SELECT g.nom, g.ville

FROM Garage g

JOIN Mecanicien m ON g.idgarage = m.idgarage

WHERE m.niveau = 2

Compléter le plan d'exécution réparti proposé ci-dessous,  tel que :
*   site **S2** : M = sous-requête sur la table Mecanicien,
*   transférer le résultat de M vers le site S1 contenant Garage,
*   site **S1** : R1 = sous-requête avec jointure,
*   transférer le résultat de R1 vers S0.

In [ ]:
M = S2.execute("...........................").df()
M_at_S2  = os.path.join(local_dir, ".........", "M")
M.to_csv(..........., index=False)
M

Loading ITables v2.7.3 from the internet... (need help?)


Transférer le résultat de la requête M depuis le site S2 vers le site S1

In [ ]:
print("transfert vers S1, nombre de tuples:", len(M))
M=None

M_at_S1 = os.path.join(local_dir, "..............", "M")
shutil.move(.........., .................)
# on considère que c'est équivalent à un transfert distant (par scp) depuis S2 vers S1

transfert vers S1, nombre de tuples: 5


'tmp/S1/M'

Importer le résultat dans le SGBD de S1

In [ ]:
# S1.execute("create or replace table T1 as select cast(idgarage as int) as idgarage from read_csv('/tmp/S1/T1', AUTO_DETECT=TRUE)")
S1.execute(f"create or replace table M as select * from read_csv(........................, AUTO_DETECT=TRUE)")
S1.execute("select * from M").df()

Loading ITables v2.7.3 from the internet... (need help?)


In [ ]:
# compléter
R1 = S1.execute("SELECT ................................").df()
R1_at_S1 = os.path.join(local_dir, "..................", "R1")
R1.to_csv(............., index=False)
R1

Loading ITables v2.7.3 from the internet... (need help?)


transférer le résultat de S1 vers S0

In [ ]:
print("transfert vers S0, nombre de tuples:", len(R1))
R1_at_S0 = os.path.join(local_dir, ..........., "R1")
R1=None
shutil.move(.............., .............)

transfert vers S0, nombre de tuples: 5


'tmp/S0/R1'

In [ ]:
S0.execute(f"SELECT * from  read_csv(.................., AUTO_DETECT=TRUE)").df()

Loading ITables v2.7.3 from the internet... (need help?)


## Question 2

Requête R2:

SELECT g.ville, g.nom, p.idpers

FROM Mecanicien m JOIN Garage g ON m.idgarage = g.idgarage JOIN Personne p ON p.idpers = m.idpers

WHERE p.age > 18

AND g.jourdefermeture = 'Dimanche'

ORDER BY g.ville, g.nom


In [ ]:
# ....


# Exercice 2

L'objectif de cet exercice est d'automatiserle traitement d'une requête de jointure entre 2 sites.


## 1) Fonction traiter_requête

Définir la fonction *traiter_requete(requete)* qui implante le plan d'exécution d'une requête répartie entre 2 tables A et B, telle que :
*  La requête peut contenir des sélections et des projections.
*  Une jointure entre A et B est définie dans une clause JOIN ON.
*  A et B ne sont pas sur le même site.
*  On commence par interroger le site **S1** pour transférer les données utiles vers **S2**.
*  penser à ne **pas** transférer les attributs inutilisés dans la requête.
*  La jointure est traitée sur le site **S2**.
*  Le résultat est transféré dans S0.


Indications, interroger les sites (cf requête *shema_base* ci-dessus) pour savoir où est stockée chaque table.


In [ ]:
# def traiter_requete(requete):
#   requete = Query(requete)




Appliquer la fonction sur la requête

In [ ]:
# ...

## 2) Transfert minimal

Définir la fonction *traiter_requête_opt* qui améliore la fonction ci-dessus afin de **minimiser les transferts**.

Choisir de traiter la jointure sur le site S1 ou S2 de telle sorte que cela minimise les transferts.
Envisager plusieurs solutions avec des transferts de S1 vers S2 ou de S2 vers S1 et traiter la solution avec le moins de transferts.


# Exercice 3

On propose de répartir la base **TPCH** (utilisée dans le TP précédent) tq :
*   **S1** contient les fragments de Customer dont le pays est le Canada
*   **S2**  idem mais pour le pays France

Remarque : la table Nation n'est ni sur S1 ni sur S2.


In [17]:
# scale factor
# sf = 0.2
sf = 0.05

drop_tpch_tables = """
  DROP TABLE IF EXISTS customer;
  DROP TABLE IF EXISTS lineitem;
  DROP TABLE IF EXISTS nation;
  DROP TABLE IF EXISTS orders;
  DROP TABLE IF EXISTS part;
  DROP TABLE IF EXISTS partsupp;
  DROP TABLE IF EXISTS region;
  DROP TABLE IF EXISTS supplier;
  """
S1.execute(drop_tpch_tables)

print("generate database with scale factor sf = ", sf)
S1.execute(f"CALL dbgen(sf={sf})")


drop_S1 = """
  -- on garde Customer et  Nation

  DROP TABLE IF EXISTS lineitem;
  DROP TABLE IF EXISTS orders;
  DROP TABLE IF EXISTS part;
  DROP TABLE IF EXISTS partsupp;
  DROP TABLE IF EXISTS region;
  DROP TABLE IF EXISTS supplier;
  """
S1.execute(drop_S1)


requete_globale = """
               select *
               from Customer
               join Nation on c_nationkey = n_nationkey
               where n_name in ('CANADA', 'FRANCE')
               """

# fragment Customer_Canada
requete_sur_S1 = """
               create or replace table Customer_Canada as
               select c.*
               from Customer c
               join Nation n on c_nationkey = n_nationkey
               where n_name = 'CANADA'
               order by c_acctbal
               """

S1.execute(requete_sur_S1)


# on efface Customer et Nation pour ne garder que la fragment Customer_Canada sur s1
drop_S1 = """
  DROP TABLE IF EXISTS Customer;
  DROP TABLE IF EXISTS Nation;
  """
S1.execute(drop_S1)

print("done")


generate database with scale factor sf =  0.05


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done


In [20]:
S1.execute("SUMMARIZE Customer_Canada").df()

Loading ITables v2.7.3 from the internet... (need help?)


## Question 1
Proposer une exécution répartie de la requête **globale** suivante  de telle sorte que la taille totale (en octets) des données transférées soit minimale.


SELECT c.*

FROM Customer c

JOIN Nation on c_nationkey = n_nationkey

WHERE n_name in ('CANADA, 'FRANCE')

ORDER BY c_acctbal

LIMIT 10


In [23]:
R1 = S1.execute("select * from Customer_Canada order by c_acctbal limit 10").df()
R2 = S2.execute("select * from Customer_France order by c_acctbal limit 10").df()

R2_at_S2  = os.path.join(local_dir, "S2", "R2")
R2_at_S1 = os.path.join(local_dir, "S1", "R2")

res = []
indice1 = 0
indice2 = 0

R2.iloc[indice2].to_csv(R2_at_S2, index=False)
shutil.move(R2_at_S2, R2_at_S1)
recu = S1.execute(f"select * from read_csv('{R2_at_S1}', AUTO_DETECT=TRUE)").df()
indice2 += 1
print(recu)

# Transférer les tuples de R2 un par un
while(len(res) < 10):
  if (R1.iloc[indice1].c_acctbal <= recu.c_acctbal):
    res.append(tuple(R1.iloc[indice1]))
    indice1 += 1
  else:
    res.append(tuple(recu[0]))

    # Mini optimisation pour ne pas recevoir un tuple en trop si on en a déjà 10.
    if (len(res) == 10):
      break
    else:
      R2.iloc[indice2].to_csv(R2_at_S2, index=False)
      shutil.move(R2_at_S2, R2_at_S1)
      recu = S1.execute(f"select * from read_csv('{R2_at_S1}', AUTO_DETECT=TRUE)").df()
      indice2 += 1

# Faire la jointure avec Nation

# Create an in-memory table from the list of tuples (res)
res_df = pd.DataFrame(res, columns=R1.columns)  # Ensure that the columns match the structure
res_df_path = os.path.join(local_dir, "S1", "res_table.csv")
res_df.to_csv(res_df_path, index=False)

# Now, perform the join with another table, for example, 'Nation'
join_query = f"""
CREATE OR REPLACE TABLE FinalResult AS
SELECT *
FROM read_csv('{res_df_path}', AUTO_DETECT=TRUE) AS R2_res
JOIN Nation AS N
ON R2_res.c_nationkey = N.n_nationkey
"""

# Execute the join query
S1.execute(join_query)

# Optionally, fetch the result from the database
final_result = S1.execute("SELECT * FROM FinalResult").df()
print(final_result)

FileNotFoundError: [Errno 2] No such file or directory: 'tmp/S2/R2'

## Question 2
Même question, mais pour la requête :

SELECT c_mktsegment, sum(c_acctbal) as balance

FROM Customer c

JOIN Nation on c_nationkey = n_nationkey

WHERE n_name in ('CANADA, 'FRANCE')

GROUP BY c_mktsegment


In [25]:
R1 = S1.execute("select c_mktsegment, sum(c_acctbal) from Customer_Canada GROUP BY c_mktsegment").df()
R2 = S2.execute("select c_mktsegment, sum(c_acctbal) from Customer GROUP BY c_mktsegment").df()

R1_at_S1  = os.path.join(local_dir, "S1", "R1")
R2_at_S2  = os.path.join(local_dir, "S2", "R2")
R1_at_S2  = os.path.join(local_dir, "S2", "R1")
R2_at_S1 = os.path.join(local_dir, "S1", "R2")

# Comparer les tailles de S2 et S1 et envoyer les données de S2 vers S1 (ou inversement)
if (len(R1) <= len(R2)):
  R1.to_csv(R1_at_S1, index=False)
  shutil.move(R1_at_S1, R1_at_S2)
  S2.execute(f"create or replace table R1 as select * from read_csv('{R1_at_S2}', AUTO_DETECT=TRUE)").df()
else:
  R2.to_csv(R2_at_S2, index=False)
  shutil.move(R2_at_S2, R2_at_S1)
  S1.execute(f"create or replace table R2 as select * from read_csv('{R2_at_S1}', AUTO_DETECT=TRUE)").df()

# Faire la jointure entre R1, R2 et Nation
val = "R1" if len(R1) <= len(R2) else "R2"
join_query = f"""
CREATE OR REPLACE TABLE FinalResult AS
SELECT c_mktsegment, sum(c_acctbal)
FROM Customer as C
JOIN {val} AS R
JOIN Nation AS N
ON R.c_nationkey = N.n_nationkey
GROUP BY c_mktsegment
"""

# Execute the join query
S1.execute(join_query)

# Optionally, fetch the result from the database
final_result = S1.execute("SELECT * FROM FinalResult").df()
print(final_result)

CatalogException: Catalog Error: Table with name Customer does not exist!
Did you mean "sqlite_master"?

LINE 1: select c_mktsegment, sum(c_acctbal) from Customer GROUP BY c_mktsegment
                                                 ^